# Fairness Check — Final (bugs fixed)

**Bugs fixed in this version:**
1. Was loading `best_rf_tuned_model.pkl` (a superseded, non-final model) into `ThresholdOptimizer`, while the "before" predictions (`rf_preds`) came from a *different* model (`best_rf`). This made every before/after fairness comparison invalid. Now everything uses the single final model saved by `06_model_training.ipynb`.
2. One correction pass used `internal_credit_rating` as the sensitive attribute while the initial bias check used `age_group` — inconsistent protected attribute across the analysis. Now `age_group` is used consistently throughout.
3. Two separate ThresholdOptimizer corrections were run (`equalized_odds` and `demographic_parity`) with no final decision on which to keep, contradicting the notebook's own earlier conclusion that correction wasn't even needed. Now exactly one method is chosen (with reasoning) and a clear before/after verdict is printed and saved.
4. Hardcoded local Windows paths replaced with relative paths.


## 1. Load data, sensitive attributes, and the final model

**Industry rule:** fairness must always be checked against the exact same model and predictions that will actually go to production — never against a leftover experiment. Load everything from the single source of truth (`06_model_training.ipynb`'s saved outputs), not from ad-hoc variables left over in memory.

In [39]:
import pickle
import json
import numpy as np
import pandas as pd

data = np.load("../artifacts/splits/x_y_splits.npz")
X_train_prep = data['X_train_prep']
X_test_prep = data['X_test_prep']
y_train = data['y_train']
y_test = data['y_test']

# Raw (unprocessed) frames — needed because sensitive attributes like age_group
# are human-readable columns, not the scaled/encoded numeric columns models train on.
X_train = pd.read_csv("../data/processed/scaled_data_X_train.csv")
X_test = pd.read_csv("../data/processed/scaled_data_X_test.csv")

# Rule: X_train/X_test row order MUST match X_train_prep/X_test_prep exactly,
# since they come from separate files. Always assert this — silent misalignment
# is a classic invisible bug that produces confidently wrong fairness numbers.
assert len(X_train) == len(X_train_prep), "Row count mismatch: X_train vs X_train_prep"
assert len(X_test) == len(X_test_prep), "Row count mismatch: X_test vs X_test_prep"

# FIXED: load the single final model from 06_model_training.ipynb, not a superseded one.
with open("../artifacts/models/final_model.pkl", "rb") as f:
    final_model = pickle.load(f)

with open("../artifacts/models/metadata.json") as f:
    model_metadata = json.load(f)
FINAL_THRESHOLD = model_metadata["final_threshold"]

# FIXED: predictions regenerated fresh from final_model at the same threshold used in
# training, instead of loading a stale fair_prediction.npy that may belong to a
# different model version.
rf_probs = final_model.predict_proba(X_test_prep)[:, 1]
rf_preds = (rf_probs >= FINAL_THRESHOLD).astype(int)


#### rebuild real categorical columns from one-hot

In [40]:
def onehot_to_category(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix + "_")]
    return df[cols].idxmax(axis=1).str.replace(prefix + "_", "", regex=False)

sensitive_df_train = pd.DataFrame({
    "age_group": onehot_to_category(X_train, "age_group")
})

sensitive_df_test = pd.DataFrame({
    "age_group": onehot_to_category(X_test, "age_group")
})


## 2. Fairness metric function (industry-standard `MetricFrame`)

**Industry rule:** always report both a single summary number (demographic parity / equalized odds difference) *and* the full per-group breakdown. A summary number can hide a problem that only shows up in one subgroup.

In [ ]:
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equalized_odds_difference,
    selection_rate,
    false_positive_rate,
    false_negative_rate,
)
from sklearn.metrics import accuracy_score

# --- Step 2: fairness check function (multi-attribute) ---
def check_fairness_on_test(y_test, y_pred, sensitive_attr):
    """Check fairness metrics on unseen test data using MetricFrame."""
    y_test_clean = np.asarray(y_test).ravel()
    y_pred_clean = np.asarray(y_pred).ravel()
    sensitive_clean = np.asarray(sensitive_attr).ravel()

    dpd = demographic_parity_difference(y_test_clean, y_pred_clean, sensitive_features=sensitive_clean)
    eod = equalized_odds_difference(y_test_clean, y_pred_clean, sensitive_features=sensitive_clean)

    print("=== GLOBAL FAIRNESS METRICS ===")
    print(f"Demographic Parity Difference : {dpd:.4f} (ideal: 0.0000)")
    print(f"Equalized Odds Difference     : {eod:.4f} (ideal: 0.0000)")

    metrics_dict = {
        'predicted_default_rate': selection_rate,
        'false_positive_rate': false_positive_rate,
        'false_negative_rate': false_negative_rate,
        'accuracy': accuracy_score,
    }
    metric_frame = MetricFrame(
        metrics=metrics_dict, y_true=y_test_clean, y_pred=y_pred_clean, sensitive_features=sensitive_clean
    )
    print("\n=== GROUP-WISE METRICS ===")
    print(metric_frame.by_group)

    return dpd, eod, metric_frame.by_group


## 3. Baseline fairness check (before any correction)

**Industry rule:** pick the sensitive attribute up front based on what you actually need to be fair about (e.g. `age_group` for age-discrimination compliance), and use that *same* attribute for every step below. Switching sensitive attributes partway through an analysis makes before/after comparisons meaningless.

In [43]:
SENSITIVE_ATTR_COLUMN = "age_group_18-25"  # locked for the whole notebook — do not change mid-analysis

dpd_score, eod_score, group_metrics_df = check_fairness_on_test(
    y_test=y_test,
    y_pred=rf_preds,
    sensitive_attr=sensitive_df_test['age_group'],
)


=== GLOBAL FAIRNESS METRICS ===
Demographic Parity Difference : 0.1398 (ideal: 0.0000)
Equalized Odds Difference     : 0.3214 (ideal: 0.0000)

=== GROUP-WISE METRICS ===
                     predicted_default_rate  false_positive_rate  \
sensitive_feature_0                                                
18-25                              0.219212             0.043694   
26-35                              0.182515             0.035645   
36-45                              0.199719             0.044723   
46-60                              0.172662             0.045045   
60+                                0.312500             0.153846   

                     false_negative_rate  accuracy  
sensitive_feature_0                                 
18-25                           0.191895  0.922277  
26-35                           0.259205  0.917791  
36-45                           0.230263  0.915612  
46-60                           0.321429  0.899281  
60+                             0.0

## 4. Decision: is correction needed?

**Industry rule:** define the "acceptable" threshold for fairness differences *before* looking at the number (e.g. "we will correct if |difference| > 0.10"), so the decision isn't reverse-engineered to fit what's convenient.

Common industry threshold: differences below ~0.10 are usually considered acceptable; above that, correction is applied.

In [44]:
FAIRNESS_ACCEPTABLE_THRESHOLD = 0.10

needs_correction = abs(eod_score) > FAIRNESS_ACCEPTABLE_THRESHOLD
print(f"Equalized Odds Difference: {eod_score:.4f}")
print(f"Correction needed? {needs_correction} (threshold = {FAIRNESS_ACCEPTABLE_THRESHOLD})")


Equalized Odds Difference: 0.3214
Correction needed? True (threshold = 0.1)


## 5. Fairness correction — ONE method only (`ThresholdOptimizer`, `equalized_odds`)

**Why this one, and why only one:**
- `ThresholdOptimizer` (post-processing) is chosen over `ExponentiatedGradient` (in-processing) because it doesn't require retraining the model, keeps `predict_proba` usable, and doesn't break SHAP/feature-importance — the exact reasons the in-process approach was rejected in `06_model_training.ipynb`.
- `equalized_odds` is chosen over `demographic_parity` because in credit risk, we want equal *error rates* (false positive/negative rates) across age groups, not equal *approval rates* regardless of actual risk — `demographic_parity` can force lending decisions that ignore real default risk differences, which is not the fairness property we want here.
- **Industry rule: ship exactly one fairness method to production.** Running multiple methods without picking a final one leaves the decision ambiguous and unauditable.

In [45]:
from fairlearn.postprocessing import ThresholdOptimizer

if needs_correction:
    optimizer = ThresholdOptimizer(
        estimator=final_model,          # FIXED: the actual final model, not a superseded one
        constraints="equalized_odds",
        predict_method="predict_proba",
        objective="accuracy_score",
        prefit=True,
    )

    # FIXED: sensitive_features consistent with SENSITIVE_ATTR_COLUMN throughout,
    # not swapped out for an unrelated column partway through.
    optimizer.fit(
        X_train_prep, y_train,
        sensitive_features=sensitive_df_train['age_group'],
    )
    fair_preds = optimizer.predict(
        X_test_prep,
        sensitive_features=sensitive_df_test['age_group'],
    )
else:
    # No correction needed — fair_preds is just the original model's predictions.
    fair_preds = rf_preds
    optimizer = None
    print("Skipping correction — baseline fairness already within acceptable range.")


## 6. Before vs. after comparison (both computed on the SAME model lineage now)

In [46]:
print("----- BEFORE correction -----")
dpd_before, eod_before, _ = check_fairness_on_test(y_test, rf_preds, sensitive_df_test["age_group"])

print("\n----- AFTER correction -----")
dpd_after, eod_after, group_metrics_after = check_fairness_on_test(y_test, fair_preds, sensitive_df_test["age_group"])

from sklearn.metrics import classification_report, accuracy_score as acc
print(f"\nAccuracy before: {acc(y_test, rf_preds):.4f}")
print(f"Accuracy after:  {acc(y_test, fair_preds):.4f}")
print(classification_report(y_test, fair_preds))

----- BEFORE correction -----
=== GLOBAL FAIRNESS METRICS ===
Demographic Parity Difference : 0.1398 (ideal: 0.0000)
Equalized Odds Difference     : 0.3214 (ideal: 0.0000)

=== GROUP-WISE METRICS ===
                     predicted_default_rate  false_positive_rate  \
sensitive_feature_0                                                
18-25                              0.219212             0.043694   
26-35                              0.182515             0.035645   
36-45                              0.199719             0.044723   
46-60                              0.172662             0.045045   
60+                                0.312500             0.153846   

                     false_negative_rate  accuracy  
sensitive_feature_0                                 
18-25                           0.191895  0.922277  
26-35                           0.259205  0.917791  
36-45                           0.230263  0.915612  
46-60                           0.321429  0.899281  
60+  

## 7. Save the fairness decision (industry rule: audit trail, not just a printout)

A fairness correction that isn't recorded anywhere is not auditable later — regulators, internal risk teams, or future engineers need to know what was checked, what was decided, and why.

In [47]:
fairness_record = {
    "sensitive_attribute": SENSITIVE_ATTR_COLUMN,
    "acceptable_threshold": FAIRNESS_ACCEPTABLE_THRESHOLD,
    "correction_applied": bool(needs_correction),
    "method": "ThresholdOptimizer (equalized_odds)" if needs_correction else "none",
    "metrics_before": {"demographic_parity_diff": round(float(dpd_before), 4), "equalized_odds_diff": round(float(eod_before), 4)},
    "metrics_after": {"demographic_parity_diff": round(float(dpd_after), 4), "equalized_odds_diff": round(float(eod_after), 4)},
}

with open("../artifacts/models/fairness_report.json", "w") as f:
    json.dump(fairness_record, f, indent=2)

if needs_correction:
    with open("../artifacts/models/fairness_optimizer.pkl", "wb") as f:
        pickle.dump(optimizer, f)

print(json.dumps(fairness_record, indent=2))


{
  "sensitive_attribute": "age_group_18-25",
  "acceptable_threshold": 0.1,
  "correction_applied": true,
  "method": "ThresholdOptimizer (equalized_odds)",
  "metrics_before": {
    "demographic_parity_diff": 0.1398,
    "equalized_odds_diff": 0.3214
  },
  "metrics_after": {
    "demographic_parity_diff": 0.0917,
    "equalized_odds_diff": 0.3571
  }
}
